<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/18_classification/18_6_Ensemble/18_6_0_Intro_Ensemble.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ensemble Methods: A Deeper Dive

---

## Introduction: Building on What We Know

In notebook 17_2_2 we were introduced to tree-based methods (decision trees, random forests, gradient boosting, and XGBoost) and saw how they performed on the Ames Housing dataset, learning about overfitting, feature importance, and the bias-variance tradeoff along the way. The 18_1 series then put XGBoost to work as a classifier.

This series (18_6) covers these methods **in more detail**. We will:

1.  **Examine each ensemble method separately**, so that it is clear *how* bagging, random forests, and boosting work.
2.  **Focus on classification-specific concerns**, including precision, recall, false negatives in medical diagnosis, and ROC curves.
3.  **Tune hyperparameters systematically**, using GridSearchCV to find good settings rather than guessing.
4.  **Compare all five methods directly** (single tree, bagging, random forest, boosting, and BART), using nested cross-validation for unbiased estimates.

We'll use the **Wisconsin Breast Cancer dataset**, a medical diagnosis problem where the cost of a false negative (missing a cancer) is far higher than the cost of a false positive (an unnecessary biopsy). That makes it a good dataset for studying the precision-recall tradeoff in a realistic setting.

### Recommended Preparation
- [StatQuest: Random Forests](https://www.youtube.com/watch?v=J4Wdy0Wc_xQ&t=143s): a clear visual explanation of the core concepts
- Review notebook 17_2_2 (regression trees on Ames) if you need a refresher on tree-based methods

## The Ensemble Hierarchy

All ensemble methods share the same core idea: **combine many models so that the combination predicts better than any one of them**. They differ in *how* they build and combine those models. Boosting combines many weak models (shallow trees), while bagging and random forests average many strong but unstable models (deep trees).

| Method | How Trees Are Built | How Predictions Are Combined | Primary Goal |
|---|---|---|---|
| **Single Tree** | One tree, grown top-down | N/A | Interpretability |
| **Bagging** | Independent trees on bootstrap samples | Majority vote / average | Reduce variance |
| **Random Forest** | Independent trees + random feature subsets | Majority vote / average | Reduce variance + decorrelate |
| **Boosting** | Sequential trees, each correcting errors | Weighted sum | Reduce bias |
| **BART** | Bayesian tree perturbation | Posterior distribution | Reduce both + quantify uncertainty |

### Variance vs. Bias

- **Bagging and Random Forests** reduce **variance**. They make unstable models (deep trees) more stable by averaging many of them.
- **Boosting** reduces **bias**. It takes weak models (shallow trees) and makes them stronger by sequentially correcting their mistakes.
- **BART** addresses **both** while also providing uncertainty estimates (credible intervals).

In the notebooks that follow, we'll see each of these in action.

## The Dataset: Wisconsin Breast Cancer

We'll use the Wisconsin Diagnostic Breast Cancer (WDBC) dataset throughout this series. It contains measurements from digitized images of fine needle aspirates of breast masses.

- **569 samples**, **30 features** (mean, standard error, and "worst" values for 10 cell nucleus characteristics)
- **Binary target**: Malignant (1) or Benign (0)
- **Class distribution**: ~63% benign, ~37% malignant, which is moderately imbalanced

This is a **high-stakes classification problem**. A false negative (predicting benign when the tumor is malignant) means a cancer goes undiagnosed. A false positive (predicting malignant when the tumor is benign) means an unnecessary biopsy. These costs are not equal, which is why we need to look beyond accuracy.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
df = data.frame

# sklearn codes malignant as 0 and benign as 1. We flip it so that 1 = malignant,
# the class we want to detect, and use this encoding throughout the 18_6 series.
df['target'] = 1 - df['target']

print(f"Dataset shape: {df.shape}")
print(f"\nClass distribution:")
print(df['target'].value_counts())
print(f"\nClass proportions:")
print(df['target'].value_counts(normalize=True))
print(f"\nFeatures: {len(data.feature_names)}")
print("Target names: 1 = Malignant, 0 = Benign")

Dataset shape: (569, 31)

Class distribution:
target
0    357
1    212
Name: count, dtype: int64

Class proportions:
target
0    0.627417
1    0.372583
Name: proportion, dtype: float64

Features: 30
Target names: 1 = Malignant, 0 = Benign


### Why This Dataset Matters

The 37% malignant rate means a naive model that always predicts "benign" would achieve 63% accuracy. Any useful model must do substantially better than this baseline.

Accuracy alone is also not enough. In the next notebook, a single decision tree reaches 92% test accuracy while missing 14 of the 64 malignant tumours in the test set (about 1 in 5), which would make it unsuitable for clinical use. We need to evaluate precision, recall, and the ROC curve, which we covered in 18_1_1 and 18_1_2.

In the next notebook, we'll start with the building block: a single decision tree.